## Libraries Installation

In [ ]:
!pip install azure-storage-blob
!pip install snowflake-connector-python
!pip install snowflake-sqlalchemy
!pip install pandas
!pip install rapidfuzz
!pip install sqlalchemy

## Libraries Import

In [1]:
import sys
import subprocess

# Install all required packages
packages = [
    'azure-storage-blob',
    'snowflake-connector-python',
    'snowflake-sqlalchemy',
    'rapidfuzz',
    'sqlalchemy',
    'pandas',
    'numpy'
]

for package in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', package], check=True)

print('All packages installed!')

# Imports
import pandas as pd
import numpy as np
import json
import io
import warnings
warnings.filterwarnings('ignore')
from rapidfuzz import fuzz, process
from azure.storage.blob import BlobServiceClient
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine, text

print('All libraries imported successfully!')

All packages installed!
All libraries imported successfully!


## Connection String Load

In [3]:
def load_config_azure(config_path='config(1).json'):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    return config['AZURE_CONNECTION_STRING'], config['CONTAINER_NAME']

def load_config_snowflake(config_path='config.json'):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    return (
        config['SNOWFLAKE_USER'],
        config['SNOWFLAKE_PASSWORD'],
        config['SNOWFLAKE_ACCOUNT'],
        config['SNOWFLAKE_WAREHOUSE'],
        config['SNOWFLAKE_DATABASE'],
        config['SNOWFLAKE_SCHEMA']
    )

# Load Azure credentials
AZURE_CONNECTION_STRING, CONTAINER_NAME = load_config_azure('config.json')
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
print('Azure connected successfully')

# Load Snowflake credentials
SF_USER, SF_PASSWORD, SF_ACCOUNT, SF_WAREHOUSE, SF_DATABASE, SF_SCHEMA = load_config_snowflake('config.json')
engine = create_engine(URL(
    user=SF_USER,
    password=SF_PASSWORD,
    account=SF_ACCOUNT,
    warehouse=SF_WAREHOUSE,
    database=SF_DATABASE,
    schema=SF_SCHEMA
))
print('Snowflake connected successfully')

ValueError: Connection string missing required connection details.

## Extraction

### Extraction - RateMD Doctors JSON from Azure

In [ ]:

RATEMD_CONTAINER = 'ratemdreviews'

print('Downloading doctors_flat.json from Azure...')
blob_client = blob_service_client.get_blob_client(
    container=RATEMD_CONTAINER,
    blob='doctors_flat.json'
)
blob_data = blob_client.download_blob().readall()
print(f'Downloaded {len(blob_data) / 1024 / 1024:.1f} MB')

# Parse JSON
doctors_raw = pd.read_json(io.BytesIO(blob_data))
print('Raw shape:', doctors_raw.shape)
print('Columns:', doctors_raw.columns.tolist())
doctors_raw.head(3)

## Transformation

### Data Profiling

In [ ]:
print('=== RATEMD RAW ===')
print('Shape:', doctors_raw.shape)
print()
print('Dtypes:')
print(doctors_raw.dtypes)
print()
print('Null counts:')
print(doctors_raw.isnull().sum())
print()
print('Sample row:')
print(doctors_raw.iloc[0])

### Data Cleaning

In [ ]:
# Step 1: Filter to US doctors only
print('Before US filter:', len(doctors_raw))
df = doctors_raw[doctors_raw['country'] == 'us'].copy()
print('After US filter:', len(df))
print()

In [ ]:
# Step 2: Drop rows with no name or no rating
df = df.dropna(subset=['full_name'])
print('After dropping null names:', len(df))

# Step 3: Standardize doctor names - strip whitespace, title case
df['full_name_clean'] = df['full_name'].str.strip().str.title()

# Step 4: Drop rows with no postal code (needed for geography join)
df = df.dropna(subset=['postal_code'])
print('After dropping null postal codes:', len(df))

# Step 5: Standardize postal code to 5 digits
df['postal_code'] = df['postal_code'].astype(str).str.strip().str[:5].str.zfill(5)

print()
print('Sample cleaned data:')
df[['full_name_clean', 'specialty', 'postal_code', 'rating_average', 'rating_count']].head(10)

In [ ]:
# Step 6: Fuzzy matching deduplication
# Identify near-duplicate doctor names (same name spelled slightly differently)
# We group by postal_code + specialty first to narrow comparisons
print('Running fuzzy matching deduplication...')
print('This may take a few minutes...')

from rapidfuzz import fuzz

def get_canonical_name(name, name_list, threshold=90):
    """Find the best matching canonical name for a given name."""
    if not name_list:
        return name
    best_match = process.extractOne(name, name_list, scorer=fuzz.token_sort_ratio)
    if best_match and best_match[1] >= threshold:
        return best_match[0]
    return name

# Build canonical name map per specialty+zip group
canonical_map = {}
groups = df.groupby(['postal_code', 'specialty'])['full_name_clean'].apply(list)

for (zip_code, specialty), names in groups.items():
    unique_names = list(set(names))
    canonical_names = []
    for name in unique_names:
        canonical = get_canonical_name(name, canonical_names)
        if canonical == name:
            canonical_names.append(name)
        canonical_map[(zip_code, specialty, name)] = canonical

# Apply canonical names
df['canonical_name'] = df.apply(
    lambda row: canonical_map.get(
        (row['postal_code'], row['specialty'], row['full_name_clean']),
        row['full_name_clean']
    ), axis=1
)

print('Fuzzy matching complete')
print('Sample canonical names:')
print(df[['full_name_clean', 'canonical_name']].head(10))

### Data Reformatting

In [ ]:
# Reformat rating columns to float
rating_cols = ['rating_average', 'rating_count', 'rating_staff',
               'rating_punctuality', 'rating_helpfulness', 'rating_knowledge']

for col in rating_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print('Rating columns reformatted')
print(df[rating_cols].describe())

### Data Transformation - Build Dimension Tables

In [ ]:
# Build dim_specialty
specialties = df[['specialty', 'specialty_name']].drop_duplicates().dropna()
specialties = specialties.reset_index(drop=True)
specialties.columns = ['specialty_slug', 'specialty_name']

dim_specialty = pd.DataFrame({
    'specialty_id':       range(1, len(specialties) + 1),
    'specialty_name':     specialties['specialty_name'],
    'specialty_group':    specialties['specialty_slug'],
    'specialty_category': specialties['specialty_slug'].str.replace('-', ' ').str.title(),
    'is_primary_care':    specialties['specialty_slug'].isin([
                              'family-medicine', 'internal-medicine',
                              'pediatrics', 'general-practice'
                          ])
})

print('dim_specialty shape:', dim_specialty.shape)
dim_specialty.head(10)

In [ ]:
# Build dim_doctor
# One row per unique canonical doctor (deduplicated)
doctor_agg = df.groupby(['canonical_name', 'specialty']).agg(
    accepting_new_patients=('accepting_patients', 'first'),
    verified=('verified', 'first')
).reset_index()

dim_doctor = pd.DataFrame({
    'doctor_id':            range(1, len(doctor_agg) + 1),
    'canonical_name':       doctor_agg['canonical_name'],
    'gender':               None,
    'years_experience':     None,
    'languages_spoken':     None,
    'practice_setting':     None,
    'accepting_new_patients': doctor_agg['accepting_new_patients']
})

print('dim_doctor shape:', dim_doctor.shape)
dim_doctor.head(10)

In [ ]:
# Build doctor count by zip, county, state
# Join df to geography to get county and state per zip
dim_geo = pd.read_sql('SELECT zip_code, county, state, state_abbr FROM dim_geography', engine)

df_geo = df.merge(dim_geo, left_on='postal_code', right_on='zip_code', how='left')

# Count unique doctors per zip code
doctors_per_zip = df_geo.groupby('postal_code')['canonical_name'].nunique().reset_index()
doctors_per_zip.columns = ['zip_code', 'doctor_count']

# Count unique doctors per county
doctors_per_county = df_geo.groupby(['county', 'state'])['canonical_name'].nunique().reset_index()
doctors_per_county.columns = ['county', 'state', 'doctor_count']

# Count unique doctors per state
doctors_per_state = df_geo.groupby('state')['canonical_name'].nunique().reset_index()
doctors_per_state.columns = ['state', 'doctor_count']

print('Doctors per zip (sample):')
print(doctors_per_zip.head())
print()
print('Doctors per state (top 10):')
print(doctors_per_state.sort_values('doctor_count', ascending=False).head(10))

### Data Transformation - Build Fact Table

In [ ]:
# Build fact_doctor_rating
# Join to dim_doctor to get doctor_id
df_fact = df.merge(
    dim_doctor[['doctor_id', 'canonical_name']],
    on='canonical_name',
    how='left'
)

# Join to dim_specialty to get specialty_id
df_fact = df_fact.merge(
    dim_specialty[['specialty_id', 'specialty_group']],
    left_on='specialty',
    right_on='specialty_group',
    how='left'
)

# Join to dim_geography to get geography_id
dim_geo_full = pd.read_sql('SELECT geography_id, zip_code FROM dim_geography', engine)
df_fact = df_fact.merge(
    dim_geo_full,
    left_on='postal_code',
    right_on='zip_code',
    how='left'
)

# Assign quality band based on avg rating
def assign_quality_band(rating):
    if rating >= 4.5: return 1   # Excellent
    elif rating >= 4.0: return 2  # Good
    elif rating >= 3.0: return 3  # Average
    elif rating >= 2.0: return 4  # Below Average
    elif rating >= 1.0: return 5  # Poor
    else: return 6                # Unrated

df_fact['quality_band_id'] = df_fact['rating_average'].apply(assign_quality_band)

# Build final fact table
fact_doctor_rating = pd.DataFrame({
    'fact_rating_id':    range(1, len(df_fact) + 1),
    'doctor_id':         df_fact['doctor_id'],
    'geography_id':      df_fact['geography_id'],
    'specialty_id':      df_fact['specialty_id'],
    'date_id':           1,  # placeholder - no review date in this dataset
    'quality_band_id':   df_fact['quality_band_id'],
    'avg_rating':        df_fact['rating_average'],
    'review_count':      df_fact['rating_count'],
    'staff_rating':      df_fact.get('rating_staff', 0),
    'punctuality_rating': df_fact.get('rating_punctuality', 0),
    'helpfulness_rating': df_fact.get('rating_helpfulness', 0),
    'knowledge_rating':  df_fact.get('rating_knowledge', 0),
    'source_doctor_id':  df_fact['doctor_id'].astype(str)
})

print('fact_doctor_rating shape:', fact_doctor_rating.shape)
print('Null counts:')
print(fact_doctor_rating.isnull().sum())
fact_doctor_rating.head()

### Data Consolidation

In [ ]:
# Final review of all tables before loading
print('=== dim_specialty ===')
print('Shape:', dim_specialty.shape)
print(dim_specialty.head(3))
print()
print('=== dim_doctor ===')
print('Shape:', dim_doctor.shape)
print(dim_doctor.head(3))
print()
print('=== fact_doctor_rating ===')
print('Shape:', fact_doctor_rating.shape)
print(fact_doctor_rating.head(3))
print()
print('=== Doctor counts by state ===')
print(doctors_per_state.sort_values('doctor_count', ascending=False).head(10))

## Loading

In [ ]:
# Create dim_specialty table
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS dim_specialty'))
    conn.execute(text("""
        CREATE TABLE dim_specialty (
            specialty_id       INTEGER,
            specialty_name     VARCHAR(200),
            specialty_group    VARCHAR(200),
            specialty_category VARCHAR(200),
            is_primary_care    BOOLEAN
        )
    """))
    conn.commit()
print('dim_specialty table created')

dim_specialty.to_sql('dim_specialty', engine, if_exists='append', index=False, method='multi')
print('dim_specialty loaded:', len(dim_specialty), 'rows')

In [ ]:
# Create dim_doctor table
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS dim_doctor'))
    conn.execute(text("""
        CREATE TABLE dim_doctor (
            doctor_id              INTEGER,
            canonical_name         VARCHAR(300),
            gender                 VARCHAR(20),
            years_experience       INTEGER,
            languages_spoken       VARCHAR(300),
            practice_setting       VARCHAR(100),
            accepting_new_patients BOOLEAN
        )
    """))
    conn.commit()
print('dim_doctor table created')

chunksize = 5000
for i in range(0, dim_doctor.shape[0], chunksize):
    chunk = dim_doctor[i:i + chunksize]
    chunk.to_sql('dim_doctor', engine, if_exists='append', index=False, method='multi')
    print(f'Loaded rows {i} to {i + len(chunk)}')
print('dim_doctor loaded successfully!')

In [ ]:
# Create fact_doctor_rating table
with engine.connect() as conn:
    conn.execute(text('DROP TABLE IF EXISTS fact_doctor_rating'))
    conn.execute(text("""
        CREATE TABLE fact_doctor_rating (
            fact_rating_id      INTEGER,
            doctor_id           INTEGER,
            geography_id        INTEGER,
            specialty_id        INTEGER,
            date_id             INTEGER,
            quality_band_id     INTEGER,
            avg_rating          FLOAT,
            review_count        INTEGER,
            staff_rating        FLOAT,
            punctuality_rating  FLOAT,
            helpfulness_rating  FLOAT,
            knowledge_rating    FLOAT,
            source_doctor_id    VARCHAR(100)
        )
    """))
    conn.commit()
print('fact_doctor_rating table created')

chunksize = 5000
for i in range(0, fact_doctor_rating.shape[0], chunksize):
    chunk = fact_doctor_rating[i:i + chunksize]
    chunk.to_sql('fact_doctor_rating', engine, if_exists='append', index=False, method='multi')
    print(f'Loaded rows {i} to {i + len(chunk)}')
print('fact_doctor_rating loaded successfully!')

In [ ]:
# Verify all tables loaded in Snowflake
tables = ['dim_specialty', 'dim_doctor', 'fact_doctor_rating']
for table in tables:
    result = pd.read_sql(f'SELECT COUNT(*) AS total FROM {table}', engine)
    print(f'{table}: {result["total"][0]} rows')

print()
print('=== Sample fact_doctor_rating ===')
sample = pd.read_sql('SELECT * FROM fact_doctor_rating LIMIT 5', engine)
print(sample)